In [9]:
import os
import cdsapi
import requests
import xarray as xr
from bs4 import BeautifulSoup

### ERA5 Surface Winds

In [2]:
dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "year": [
        "1940", "1941", "1942",
        "1943", "1944", "1945",
        "1946", "1947", "1948",
        "1949", "1950", "1951",
        "1952", "1953", "1954",
        "1955", "1956", "1957",
        "1958", "1959", "1960",
        "1961", "1962", "1963",
        "1964", "1965", "1966",
        "1967", "1968", "1969",
        "1970", "1971", "1972",
        "1973", "1974", "1975",
        "1976", "1977", "1978",
        "1979", "1980", "1981",
        "1982", "1983", "1984",
        "1985", "1986", "1987",
        "1988", "1989", "1990",
        "1991", "1992", "1993",
        "1994", "1995", "1996",
        "1997", "1998", "1999",
        "2000", "2001", "2002",
        "2003", "2004", "2005",
        "2006", "2007", "2008",
        "2009", "2010", "2011",
        "2012", "2013", "2014",
        "2015", "2016", "2017",
        "2018", "2019", "2020",
        "2021", "2022", "2023",
        "2024", "2025", "2026"
    ],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "zip",
    "area": [90, -180, 40, 180]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-06-23 13:29:25,998 INFO Request ID is 8d80ad0a-a22f-42d3-9cbb-391bf24e102d
2026-06-23 13:29:29,985 INFO status has been updated to accepted
2026-06-23 13:29:42,868 INFO status has been updated to running
2026-06-23 13:33:50,068 INFO status has been updated to successful


29d58e0905d88ad287965bcd155b6536.zip:   0%|          | 0.00/1.08G [00:00<?, ?B/s]

'29d58e0905d88ad287965bcd155b6536.zip'

### ERA5 Sea Level Pressure

In [3]:
dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": ["mean_sea_level_pressure"],
    "year": [
        "1940", "1941", "1942",
        "1943", "1944", "1945",
        "1946", "1947", "1948",
        "1949", "1950", "1951",
        "1952", "1953", "1954",
        "1955", "1956", "1957",
        "1958", "1959", "1960",
        "1961", "1962", "1963",
        "1964", "1965", "1966",
        "1967", "1968", "1969",
        "1970", "1971", "1972",
        "1973", "1974", "1975",
        "1976", "1977", "1978",
        "1979", "1980", "1981",
        "1982", "1983", "1984",
        "1985", "1986", "1987",
        "1988", "1989", "1990",
        "1991", "1992", "1993",
        "1994", "1995", "1996",
        "1997", "1998", "1999",
        "2000", "2001", "2002",
        "2003", "2004", "2005",
        "2006", "2007", "2008",
        "2009", "2010", "2011",
        "2012", "2013", "2014",
        "2015", "2016", "2017",
        "2018", "2019", "2020",
        "2021", "2022", "2023",
        "2024", "2025", "2026"
    ],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "zip",
    "area": [90, -180, 40, 180]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-06-23 13:36:49,275 INFO Request ID is 35d7f414-7152-4f3d-9be0-1660a5f195d3
2026-06-23 13:36:49,464 INFO status has been updated to accepted
2026-06-23 13:37:11,860 INFO status has been updated to running
2026-06-23 13:39:43,394 INFO status has been updated to successful


995e6aaaadbb11d7dad605bb8b192b48.zip:   0%|          | 0.00/348M [00:00<?, ?B/s]

'995e6aaaadbb11d7dad605bb8b192b48.zip'

### NSIDC CDR SIC

In [10]:
# Settings
BASE_URL = "https://noaadata.apps.nsidc.org/NOAA/G02202_V6/north/monthly/"
DOWNLOAD_DIR = "NSIDC_CDR_SIC_Monthly_1978_2026"
OUTPUT_FILE = "NSIDC_CDR_SIC_Monthly_1978_2026.nc"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Get list of .nc files
print("Reading directory listing...")
html = requests.get(BASE_URL).text
soup = BeautifulSoup(html, "html.parser")

files = []
for link in soup.find_all("a"):
    href = link.get("href", "")
    if href.endswith(".nc"):
        files.append(href)

files = sorted(files)
print(f"\nFound {len(files)} netCDF files.\n")

# Download files
for fname in files:

    url = BASE_URL + fname
    outfile = os.path.join(DOWNLOAD_DIR, fname)

    if os.path.exists(outfile):
        print(f"Exists: {fname}")
        continue

    print(f"Downloading: {fname}...")

    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(outfile, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# Remove aggregate file if present
monthly_files = sorted([
    os.path.join(DOWNLOAD_DIR, f)
    for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".nc")
])

print(f"\nCombining {len(monthly_files)} monthly files...")

# Extract SIC from each file individually
sic_list = []
for i, f in enumerate(monthly_files):
    ds = xr.open_dataset(f)
    sic = ds["cdr_seaice_conc_monthly"]

    # Remove flags
    sic = sic.where((sic >= 0) & (sic <= 100))

    # Load into memory and close file
    sic = sic.load()
    ds.close()

    sic_list.append(sic)

# Concatenate along existing time dimension
sic = xr.concat(sic_list, dim="time")

# Ensure dimensions are correct
sic = sic.transpose("time", "y", "x")

out_ds = xr.Dataset({"sic": sic})

# Save
encoding = {"sic": {"zlib": True, "complevel": 4, "dtype": "float32"}}
print(f"\nSaving {OUTPUT_FILE}")
out_ds.to_netcdf(OUTPUT_FILE, encoding=encoding)

print("\nDone!")

Reading directory listing...

Found 571 netCDF files.

Downloading: sic_psn25_197811_n07_v06r00.nc...
Downloading: sic_psn25_197812_n07_v06r00.nc...
Downloading: sic_psn25_197901_n07_v06r00.nc...
Downloading: sic_psn25_197902_n07_v06r00.nc...
Downloading: sic_psn25_197903_n07_v06r00.nc...
Downloading: sic_psn25_197904_n07_v06r00.nc...
Downloading: sic_psn25_197905_n07_v06r00.nc...
Downloading: sic_psn25_197906_n07_v06r00.nc...
Downloading: sic_psn25_197907_n07_v06r00.nc...
Downloading: sic_psn25_197908_n07_v06r00.nc...
Downloading: sic_psn25_197909_n07_v06r00.nc...
Downloading: sic_psn25_197910_n07_v06r00.nc...
Downloading: sic_psn25_197911_n07_v06r00.nc...
Downloading: sic_psn25_197912_n07_v06r00.nc...
Downloading: sic_psn25_198001_n07_v06r00.nc...
Downloading: sic_psn25_198002_n07_v06r00.nc...
Downloading: sic_psn25_198003_n07_v06r00.nc...
Downloading: sic_psn25_198004_n07_v06r00.nc...
Downloading: sic_psn25_198005_n07_v06r00.nc...
Downloading: sic_psn25_198006_n07_v06r00.nc...
Downl

In [11]:
# Test
ds = xr.open_dataset('/glade/work/skygale/_data/Downscaling/NSIDC_CDR_SIC_Monthly_1978_2026.nc')
ds

<xarray.Dataset> Size: 311MB
Dimensions:  (time: 571, x: 304, y: 448)
Coordinates:
  * time     (time) datetime64[ns] 5kB 1978-11-01 1978-12-01 ... 2026-05-01
  * x        (x) float64 2kB -3.838e+06 -3.812e+06 ... 3.712e+06 3.738e+06
  * y        (y) float64 4kB 5.838e+06 5.812e+06 ... -5.312e+06 -5.338e+06
Data variables:
    sic      (time, y, x) float32 311MB ...